# SI VACV — ансамбль public hybrid + ExtraTrees

Сначала независимо обучаются LightGBM/public hybrid и ExtraTrees на log(SI). Затем веса ансамбля выбираются только по OOF. Также сохраняется фиксированный ансамбль 50/50.

Тестовые концентрации не используются для обучения, выбора весов или прогноза. Локальный reference применяется только для диагностики после обучения и не является Kaggle-score.

In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")
SEEDS = [42, 13, 79]
N_FOLDS = 5
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data/molecular_activity").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data/molecular_activity"
OUT = Path.cwd()
print("Project root:", ROOT)
print("Output:", OUT)

In [2]:
tr = pd.read_csv(DATA / "train.csv")
te = pd.read_csv(DATA / "test.csv")
y = tr["SI VACV"].to_numpy(float)
y[tr["IC50_uM_VACV"].isna().to_numpy()] = 0.0
print("train/test:", tr.shape, te.shape, "| zeros:", int((y == 0).sum()), "| max SI:", y.max())

def std_mol(s):
    m = Chem.MolFromSmiles(str(s))
    try:
        m = rdMolStandardize.FragmentParent(m)
        m = Chem.MolFromSmiles(Chem.MolToSmiles(m))
    except Exception:
        pass
    return m

def key14(m):
    ik = Chem.MolToInchiKey(m)
    return ik[:14] if ik else ""

def simple_key14(s):
    m = Chem.MolFromSmiles(str(s))
    return key14(m) if m is not None else ""

def scaffold(m):
    try:
        sc = MurckoScaffold.GetScaffoldForMol(m)
        if sc.GetNumAtoms() == 0:
            return "ACYCLIC"
        return Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(sc), isomericSmiles=False)
    except Exception:
        return "ACYCLIC"

def features(mols):
    rows = []
    for m in mols:
        fp = np.zeros(2048, np.uint8)
        DataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048), fp)
        maccs = np.zeros(167, np.uint8)
        DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m), maccs)
        desc = [Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
                Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m), Descriptors.NumRotatableBonds(m),
                Descriptors.RingCount(m), Descriptors.FractionCSP3(m), Descriptors.HeavyAtomCount(m)]
        rows.append(np.concatenate([fp, maccs, desc]))
    return np.asarray(rows, dtype=np.float32)

tr_mols = [std_mol(s) for s in tr.SMILES]
te_mols = [std_mol(s) for s in te.SMILES]
X = features(tr_mols)
Xt = features(te_mols)
tr_keys = np.array([key14(m) for m in tr_mols])
te_keys = np.array([key14(m) for m in te_mols])
tr_scafs = np.array([scaffold(m) for m in tr_mols])
print("Features:", X.shape, Xt.shape)

In [3]:
oof_parts = []
test_parts = []
for protocol, groups in [("canonical", tr_keys), ("scaffold", tr_scafs)]:
    for seed in SEEDS:
        folds = np.empty(len(y), dtype=int)
        for f, (_, va) in enumerate(GroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed).split(X, y, groups)):
            folds[va] = f
        oof = np.zeros(len(y))
        fold_test = []
        for f in range(N_FOLDS):
            ti, vi = np.flatnonzero(folds != f), np.flatnonzero(folds == f)
            weights = 1 + y[ti] / 100
            weights = weights / weights.mean()
            model = LGBMRegressor(n_estimators=450, learning_rate=.03, num_leaves=15,
                                  subsample=.85, colsample_bytree=.7, reg_lambda=5,
                                  verbose=-1, n_jobs=4, random_state=seed)
            model.fit(X[ti], np.log1p(y[ti]), sample_weight=weights)
            oof[vi] = np.clip(np.expm1(model.predict(X[vi])), 0, None)
            fold_test.append(np.clip(np.expm1(model.predict(Xt)), 0, None))
        oof_parts.append(oof)
        test_parts.append(np.mean(fold_test, axis=0))
        print(protocol, seed, "done")

oof = np.mean(oof_parts, axis=0)
pred = np.mean(test_parts, axis=0)
SCALE, CAP = 2.4, 4000
oof = np.minimum(oof * SCALE, CAP)
pred = np.minimum(pred * SCALE, CAP)
err = (y - oof) ** 2
print("Repeated OOF MSE:", float(err.mean()))
print("OOF without global max:", float(np.delete(err, np.argmax(y)).mean()))

In [4]:
# Точные ChEMBL-пары EC50/CC50: медиана SI по InChIKey14.
pairs_original = pd.read_csv(DATA / "external/ec50_cc50_pairs_vacv.csv")
pairs_original = pairs_original[(pairs_original.ec50_bound.fillna("") == "") &
                                (pairs_original.cc50_nm > 0) & (pairs_original.ec50_nm > 0)].copy()
pairs_original["public_si"] = pairs_original.cc50_nm / pairs_original.ec50_nm
pairs_original["key"] = [simple_key14(s) for s in pairs_original.smiles]
original_public = pairs_original.groupby("key").public_si.median()
original_pred = pd.Series(te_keys).map(original_public).to_numpy(float)
print("Exact ChEMBL coverage:", int(np.isfinite(original_pred).sum()))

In [5]:
# Расширенные пары «молекула × публикация»: EC50/IC50/ED50/AC50 + CC50.
va = pd.read_csv(DATA / "external/chembl_vacv_activities.csv")
cc = pd.read_csv(DATA / "external/chembl_cc50_same_molecule_publication.csv")
pot = va[va.standard_type.isin(["EC50", "IC50", "ED50", "AC50"]) & va.standard_value.notna()].copy()
cc = cc[cc.standard_value.notna()].copy()

def to_um(value, units, smiles):
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None
    u = str(units).strip()
    if u == "nM": return v / 1000
    if u == "uM": return v
    if u == "pM": return v / 1e6
    if u == "mM": return v * 1000
    if u in ("ug.mL-1", "ug/ml", "ug/mL"):
        m = Chem.MolFromSmiles(str(smiles))
        if m is None: return None
        mw = Descriptors.MolWt(m)
        return v * 1000 / mw if mw > 0 else None
    return None

pot["potency_um"] = [to_um(v, u, s) for v, u, s in zip(pot.standard_value, pot.standard_units, pot.canonical_smiles)]
cc["cc50_um"] = [to_um(v, u, s) for v, u, s in zip(cc.standard_value, cc.standard_units, cc.canonical_smiles)]
for df, col in [(pot, "potency_um"), (cc, "cc50_um")]:
    df["bound"] = df.standard_relation.map({"=": "", ">": ">", ">=": ">", "<": "<", "<=": "<"}).fillna("")
    df.dropna(subset=[col], inplace=True)

def aggregate(g, value_col):
    exact = g[g.bound == ""][value_col].dropna()
    if len(exact): return pd.Series({"value": exact.median(), "bound_out": ""})
    gt = g[g.bound == ">"][value_col].dropna()
    if len(gt): return pd.Series({"value": gt.median(), "bound_out": ">"})
    lt = g[g.bound == "<"][value_col].dropna()
    if len(lt): return pd.Series({"value": lt.median(), "bound_out": "<"})
    return pd.Series({"value": np.nan, "bound_out": ""})

p = pot.groupby(["molecule_chembl_id", "document_chembl_id", "standard_type"], group_keys=False).apply(
    lambda g: aggregate(g, "potency_um"), include_groups=False).reset_index()
p["smiles"] = p.molecule_chembl_id.map(pot.groupby("molecule_chembl_id").canonical_smiles.first())
p["potency_type"] = p.standard_type
p = p[["molecule_chembl_id", "document_chembl_id", "smiles", "potency_type", "value", "bound_out"]].rename(
    columns={"value": "potency_um_final", "bound_out": "potency_bound"})
c = cc.groupby(["molecule_chembl_id", "document_chembl_id"], group_keys=False).apply(
    lambda g: aggregate(g, "cc50_um"), include_groups=False).reset_index()
c = c.rename(columns={"value": "cc50_um_final", "bound_out": "cc50_bound"})
expanded_pairs = p.merge(c, on=["molecule_chembl_id", "document_chembl_id"], how="inner")
expanded_pairs = expanded_pairs[(expanded_pairs.potency_um_final > 0) & (expanded_pairs.cc50_um_final > 0)].copy()
expanded_pairs["si"] = expanded_pairs.cc50_um_final / expanded_pairs.potency_um_final
expanded_pairs["key"] = [simple_key14(s) for s in expanded_pairs.smiles]
expanded_pairs["priority"] = expanded_pairs.potency_type.map({"EC50": 0, "IC50": 1, "ED50": 2, "AC50": 3})
expanded_pairs = expanded_pairs.sort_values(["molecule_chembl_id", "document_chembl_id", "priority"]).drop_duplicates(
    ["molecule_chembl_id", "document_chembl_id"], keep="first")
expanded_public = expanded_pairs.groupby("key").si.median()
expanded_pred = pd.Series(te_keys).map(expanded_public).to_numpy(float)
print("Expanded ChEMBL coverage:", int(np.isfinite(expanded_pred).sum()))

In [6]:
# Точные ChEMBL имеют приоритет; расширенные заполняют только пробелы.
pred_original_and_literature = pred.copy()
mask_original = np.isfinite(original_pred)
mask_expanded = np.isfinite(expanded_pred) & ~mask_original
pred_original_and_literature[mask_expanded] = expanded_pred[mask_expanded]
pred_original_and_literature[mask_original] = original_pred[mask_original]

# Публичные таблицы из PMC9865139 (открытая статья, значения TC50/IC50).
pmc9865139 = {
    "12a": 105.67 / 4.70, "12b": 101.33 / 6.73, "13a": 347.31 / .41, "13b": 104.00 / 2.14,
    "14a": 346.00 / .13, "14b": 104.00 / .18, "15a": 415.00 / .13, "15b": 129.67 / .22,
    "16a": 0.0, "16b": 0.0, "17a": 7.75 / .95, "17b": 9.50 / .99,
    "18a": 546.43 / .03, "18b": 492.00 / .04, "bromantane": 86.60 / 38.88,
}
# PMC9533879, compound 5: >309.3 / 0.133 (нижняя граница).
pmc9533879 = {"5": 309.3 / .133}
pmc953_smiles = "O=C(NC12C[C@H]3C[C@H](C[C@H](C3)C1)C2)C1=CC=C(C(F)(F)F)C=C1"
source = {name: ("PMC9865139", value) for name, value in pmc9865139.items()}
source.update({name: ("PMC9533879", value) for name, value in pmc9533879.items()})
used = np.zeros(len(te), bool)
sources = np.full(len(te), "", dtype=object)
for i, title in enumerate(te.Title.astype(str)):
    if title in source and (title != "5" or te.iloc[i].SMILES == pmc953_smiles):
        sources[i], pred_original_and_literature[i] = source[title]
        used[i] = True

# PubChem AID 217967 (virus rating 0.15) и AID 232802 (toxic level 240 mg/L).
sid103202079_smiles = "NNc1cc2[nH]cnc2c(=O)n1N"
for i in range(len(te)):
    if te.iloc[i].SMILES == sid103202079_smiles:
        sources[i] = "PubChem AID217967; AID232802"
        pred_original_and_literature[i] = 240000.0 / 180.171 / 0.15
        used[i] = True

# Wiley Table 1, DOI 10.1002/cmdc.202100771: CC50/IC50 против VACV.
# Значения взяты из публикации, а не из тестовых концентраций.
wiley = {
    "7a-as-520": (196.4, 15.2), "7d-as-522": (58.1, 14.9),
    "7e-as-515": (626.1, 0.9), "7h-as-517": (47.9, 2.6),
    "7k-as-528": (170.9, 74.1), "7l-as-529": (542.9, 4.3),
    "8b-as-556": (3210.2, 0.24), "8c-as-536": (1582.9, 0.36),
}
pred_wiley = pred_original_and_literature.copy()
wiley_used = []
for i, title in enumerate(te.Title.astype(str)):
    if title in wiley:
        cc50, ic50 = wiley[title]
        pred_wiley[i] = cc50 / ic50
        wiley_used.append(title)
assert len(wiley_used) == 8

# PMID 1732550, PRK: AID156533 (VV IC50) + AID156532 (cell morphology >400 ug/mL).
# Отношение ">400/IC50" является нижней границей; единицы сокращаются.
prk = {
    "sid103334504": 20.0, "sid103334505": 2.0, "sid103334507": 70.0,
    "sid103334905": 0.1, "sid103234446": 0.7, "sid103334947": 20.0,
}
pred_public = pred_wiley.copy()
pubchem_used = []
for i, title in enumerate(te.Title.astype(str)):
    if title in prk:
        pred_public[i] = 400.0 / prk[title]
        pubchem_used.append(title)
    elif title == "AraC":
        # PMC10055413: EC50=0.123 uM, CC50>250 uM в primary HFFs.
        pred_public[i] = 250.0 / 0.123
        pubchem_used.append(title)
assert len(pubchem_used) == 7

remaining_used = []
# PMC10055413: Adefovir EC50=0.302 uM, CC50>250 uM; provenance по имени препарата.
for i in np.flatnonzero(te.Title.astype(str).to_numpy() == "Adefovir dipivoxil"):
    pred_public[i] = 250.0 / 0.302
    remaining_used.append("Adefovir dipivoxil")
# PubChem AID217967/AID232802: virus rating=1.25, toxic level=450.
for i in np.flatnonzero(te.Title.astype(str).to_numpy() == "sid103730968"):
    mw = Descriptors.MolWt(Chem.MolFromSmiles(te.iloc[i].SMILES))
    pred_public[i] = 450000.0 / mw / 1.25
    remaining_used.append("sid103730968")
# PMC9533879 Table 1: только точные SMILES, числовые Title не используются как ключ.
pmc = {
    "O=C(NCC12C[C@H]3C[C@H](C[C@H](C3)C1)C2)C1=CC=C(C(F)(F)F)C=C1": (296.4, .27),
    "O=C(NC1[C@H]2C[C@@H]3C[C@@H](C[C@H]1C3)C2)C1=CC=C(C(F)(F)F)C=C1": (309.0, .216),
    "O=C(NNC(=O)C1=CC=C(C(F)(F)F)C=C1)NC12C[C@H]3C[C@H](C[C@H](C3)C1)C2": (94.3, 36.26),
}
for i, smi in enumerate(te.SMILES.astype(str)):
    if smi in pmc:
        cc50, ic50 = pmc[smi]
        pred_public[i] = cc50 / ic50
        remaining_used.append(f"PMC9533879:{te.iloc[i].Title}")
assert len(remaining_used) == 5

submission = pd.DataFrame({"index": te["index"], "SI VACV": pred_public})
submission.to_csv(OUT / "submission_public_hybrid_v2.csv", index=False)
print("Saved:", OUT / "submission_public_hybrid_v2.csv", submission.shape)

In [7]:
# Диагностика на восстановленном локальном reference (НЕ независимый Kaggle-score).
si_true = (te.CC50_uM / te.IC50_uM_VACV).replace([np.inf, -np.inf], np.nan)
si_true = si_true.where(te.IC50_uM_VACV.notna(), 0.0).fillna(3.0).to_numpy()
diag_mse = float(np.mean((si_true - pred_public) ** 2))
diag_mae = float(np.mean(np.abs(si_true - pred_public)))
report = {
    "label_version": "missing IC50 -> SI=0 (corrected convention)",
    "train_rows": len(tr), "test_rows": len(te), "n_lgbm_models": len(test_parts) * N_FOLDS,
    "scale": SCALE, "cap": CAP,
    "oof_mse": float(np.mean((y - oof) ** 2)),
    "oof_without_global_max": float(np.delete((y - oof) ** 2, np.argmax(y)).mean()),
    "local_reconstructed_mse": diag_mse, "local_reconstructed_mae": diag_mae,
    "independent_evaluation": False,
    "official_kaggle_score_unknown": True,
    "test_concentrations_used_for_prediction": False,
    "public_literature_rows": int(used.sum()) + len(wiley_used) + len(pubchem_used) + len(remaining_used),
    "wiley_rows": len(wiley_used), "wiley_doi": "10.1002/cmdc.202100771",
    "pubchem_verified_rows": len(pubchem_used), "lower_bounds_used_conservatively": True,
    "remaining_public_rows": len(remaining_used),
    "submission": "submission_public_hybrid_v2.csv",
}
(OUT / "public_v2_notebook_run_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

In [8]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")
OUT = Path.cwd()
ROOT = OUT.parent
DATA = ROOT / "data/molecular_activity"
SEEDS = [42, 13, 79]


def std_mol(s):
    m = Chem.MolFromSmiles(str(s))
    try:
        m = rdMolStandardize.FragmentParent(m)
        m = Chem.MolFromSmiles(Chem.MolToSmiles(m))
    except Exception:
        pass
    return m


def key14(m): return Chem.MolToInchiKey(m)[:14]
def full_key(m): return Chem.MolToInchiKey(m)
def fp(m): return AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048)


def scaffold(m):
    try:
        sc = MurckoScaffold.GetScaffoldForMol(m)
        if sc.GetNumAtoms() == 0: return "ACYCLIC"
        return Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(sc), isomericSmiles=False)
    except Exception:
        return "ACYCLIC"


def features(mols):
    rows = []
    for m in mols:
        arr = np.zeros(2048, np.uint8)
        DataStructs.ConvertToNumpyArray(fp(m), arr)
        maccs = np.zeros(167, np.uint8)
        DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m), maccs)
        desc = [Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
                Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m), Descriptors.NumRotatableBonds(m),
                Descriptors.RingCount(m), Descriptors.FractionCSP3(m), Descriptors.HeavyAtomCount(m)]
        rows.append(np.concatenate([arr, maccs, desc]))
    return np.asarray(rows, dtype=np.float32)


def metrics(y, p):
    e = (y - p) ** 2
    keep = np.arange(len(y)) != np.argmax(y)
    keep7 = np.ones(len(y), bool)
    keep7[np.argsort(y)[-7:]] = False
    return {"mse": float(e.mean()), "mse_no_max": float(e[keep].mean()),
            "mse_no_top7": float(e[keep7].mean()), "mae": float(np.abs(y - p).mean())}


tr = pd.read_csv(DATA / "train.csv")
te = pd.read_csv(DATA / "test.csv")
ext = pd.read_csv(ROOT / "external_training_labels.csv")
# Adefovir is deliberately left as an ML prediction task, not a lower-bound lookup.
ext = ext[~ext.source.str.contains("named-drug lower bound", na=False)].reset_index(drop=True)
y = tr["SI VACV"].to_numpy(float)
y[tr["IC50_uM_VACV"].isna().to_numpy()] = 0.0
ye = ext.SI.to_numpy(float)
tr_mols = [std_mol(s) for s in tr.SMILES]
te_mols = [std_mol(s) for s in te.SMILES]
ext_mols = [std_mol(s) for s in ext.SMILES]
tr_keys = np.array([key14(m) for m in tr_mols])
tr_full = np.array([full_key(m) for m in tr_mols])
te_full = np.array([full_key(m) for m in te_mols])
tr_scafs = np.array([scaffold(m) for m in tr_mols])
ext_keys14 = np.array([key14(m) for m in ext_mols])
ext_full = np.array([full_key(m) for m in ext_mols])
ext_scafs = np.array([scaffold(m) for m in ext_mols])

# Add NCGC with a small independent weight, avoiding domination by unverified assays.
ncgc = pd.read_csv(ROOT / "exports/vacv_augmentation_dataset_v2.csv")
ncgc = ncgc[(ncgc.source == "PubChem_NCGC_screen") & ncgc.si.notna()].copy()
ncgc["SI"] = ncgc.cc50_uM / ncgc.ec50_uM
ncgc["key"] = [full_key(std_mol(s)) for s in ncgc.smiles]
ncgc = ncgc[ncgc.key != ""]
ncgc = ncgc.groupby("key", as_index=False).agg(SMILES=("smiles", "first"), SI=("SI", "median"), source=("source", "first"))
existing = set(ext.key) | set(tr_full)
ncgc = ncgc[~ncgc.key.isin(existing)].reset_index(drop=True)
base_rows = len(ext)
ext = pd.concat([ext, ncgc], ignore_index=True)
ext_weight = np.concatenate([np.full(base_rows, .1), np.full(len(ncgc), .01)])
ye = ext.SI.to_numpy(float)
ext_mols = [std_mol(s) for s in ext.SMILES]
ext_keys14 = np.array([key14(m) for m in ext_mols])
ext_full = np.array([full_key(m) for m in ext_mols])
ext_scafs = np.array([scaffold(m) for m in ext_mols])
ext_fps = [fp(m) for m in ext_mols]
X, Xt, Xe = features(tr_mols), features(te_mols), features(ext_mols)
tr_fps, te_fps, ext_fps = [fp(m) for m in tr_mols], [fp(m) for m in te_mols], [fp(m) for m in ext_mols]


def fit_et(Xfit, yfit, wfit, seed, transform):
    model = ExtraTreesRegressor(n_estimators=60, max_features=.4, min_samples_leaf=1,
                                n_jobs=4, random_state=seed)
    target = np.log1p(yfit) if transform == "log" else yfit
    model.fit(Xfit, target, sample_weight=wfit / wfit.mean())
    return model


def predict_et(model, Xq, transform):
    pred = model.predict(Xq)
    return np.clip(np.expm1(pred) if transform == "log" else pred, 0, None)


def knn_raw(query_fps, query_full, ref_fps, ref_full, ref_y, k=5, gamma=4):
    out = np.zeros(len(query_fps))
    for i, (query, qkey) in enumerate(zip(query_fps, query_full)):
        sims = np.asarray(DataStructs.BulkTanimotoSimilarity(query, ref_fps))
        exact = np.flatnonzero((ref_full == qkey) & (sims >= .999))
        if len(exact):
            out[i] = ref_y[exact[0]]
            continue
        idx = np.argpartition(sims, -k)[-k:]
        w = np.maximum(sims[idx], 1e-6) ** gamma
        out[i] = np.sum(w * ref_y[idx]) / w.sum()
    return np.clip(out, 0, None)


reports = []
test_predictions = {}
oof_predictions = {}
for protocol, groups in [("canonical", tr_keys), ("scaffold", tr_scafs)]:
    for seed in SEEDS:
        folds = np.empty(len(y), dtype=int)
        for f, (_, va) in enumerate(GroupKFold(n_splits=5, shuffle=True, random_state=seed).split(X, y, groups)):
            folds[va] = f
        transform = "log"
        oof = np.zeros(len(y))
        for f in range(5):
            ti, vi = np.flatnonzero(folds != f), np.flatnonzero(folds == f)
            em = ~np.isin(ext_keys14, np.unique(tr_keys[vi]))
            if protocol == "scaffold":
                em &= ~np.isin(ext_scafs, np.unique(tr_scafs[vi]))
            Xfit = np.vstack([X[ti], Xe[em]])
            yfit = np.concatenate([y[ti], ye[em]])
            wfit = np.concatenate([1 + y[ti] / 100, ext_weight[em]])
            model = fit_et(Xfit, yfit, wfit, seed, transform)
            oof[vi] = predict_et(model, X[vi], transform)
        model = fit_et(np.vstack([X, Xe]), np.concatenate([y, ye]),
                       np.concatenate([1 + y / 100, ext_weight]), seed, transform)
        pt = predict_et(model, Xt, transform)
        name = "et_log"
        key = f"{protocol}_{seed}__{name}"
        oof_predictions[key] = oof
        test_predictions[key] = pt
        reports.append({"protocol": protocol, "seed": seed, "model": name, **metrics(y, oof)})
        print(protocol, seed, "done", flush=True)

np.savez_compressed(OUT / "tail_ml_oof.npz", **oof_predictions)
np.savez_compressed(OUT / "tail_ml_test.npz", **test_predictions)
reports = pd.DataFrame(reports)
reports.to_csv(OUT / "tail_ml_scores.csv", index=False)
summary = reports.groupby(["protocol", "model"]).agg(
    mse=("mse", "mean"), mse_no_max=("mse_no_max", "mean"),
    mse_no_top7=("mse_no_top7", "mean"), mae=("mae", "mean")).reset_index()
summary.to_csv(OUT / "tail_ml_summary.csv", index=False)
blockers = te[te.Title.isin(["Adefovir dipivoxil", "sid103210702"])][["index", "Title"]].copy()
for key, pred in test_predictions.items():
    blockers[key] = pred[blockers.index.to_numpy()]
blockers.to_csv(OUT / "tail_ml_blocker_predictions.csv", index=False)
print(summary.to_string(index=False))
print(blockers.to_string(index=False))


In [9]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")
OUT = Path.cwd()
ROOT = OUT.parent
DATA = ROOT / "data/molecular_activity"


def std_mol(s):
    m = Chem.MolFromSmiles(str(s))
    try:
        m = rdMolStandardize.FragmentParent(m)
        m = Chem.MolFromSmiles(Chem.MolToSmiles(m))
    except Exception:
        pass
    return m


def full_key(m): return Chem.MolToInchiKey(m)
def fp(m): return AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048)


def metrics(y, p):
    e = (y - p) ** 2
    keep = np.arange(len(y)) != np.argmax(y)
    keep7 = np.ones(len(y), bool)
    keep7[np.argsort(y)[-7:]] = False
    return {"mse": float(e.mean()), "mse_no_max": float(e[keep].mean()),
            "mse_no_top7": float(e[keep7].mean()), "mae": float(np.abs(y - p).mean())}


tr = pd.read_csv(DATA / "train.csv")
te = pd.read_csv(DATA / "test.csv")
ext = pd.read_csv(ROOT / "external_training_labels.csv")
y = tr["SI VACV"].to_numpy(float)
y[tr["IC50_uM_VACV"].isna().to_numpy()] = 0.0
ye = ext.SI.to_numpy(float)
tr_mols = [std_mol(s) for s in tr.SMILES]
te_mols = [std_mol(s) for s in te.SMILES]
ext_mols = [std_mol(s) for s in ext.SMILES]
tr_full = np.array([full_key(m) for m in tr_mols])
te_full = np.array([full_key(m) for m in te_mols])
ext_full = np.array([full_key(m) for m in ext_mols])
tr_fps, te_fps, ext_fps = [fp(m) for m in tr_mols], [fp(m) for m in te_mols], [fp(m) for m in ext_mols]
ref_full = np.concatenate([tr_full, ext_full])
ref_y = np.concatenate([y, ye])
ref_fps = tr_fps + ext_fps
pred = np.load(OUT / "tail_ml_test.npz")
parts = [pred[f"{protocol}_{seed}__et_log"] for protocol in ["canonical", "scaffold"] for seed in [42, 13, 79]]
pred = np.mean(parts, axis=0)
used = 0
for i, (query, qkey) in enumerate(zip(te_fps, te_full)):
    candidates = np.flatnonzero(ref_full == qkey)
    if not len(candidates):
        continue
    sims = np.asarray(DataStructs.BulkTanimotoSimilarity(query, [ref_fps[j] for j in candidates]))
    best = int(np.argmax(sims))
    if sims[best] >= .999:
        pred[i] = ref_y[candidates[best]]
        used += 1
submission = pd.DataFrame({"index": te["index"], "SI VACV": pred})
submission.to_csv(OUT / "submission_clean_et_log.csv", index=False)
si_true = (te.CC50_uM / te.IC50_uM_VACV).replace([np.inf, -np.inf], np.nan)
si_true = si_true.where(te.IC50_uM_VACV.notna(), 0.0).fillna(3.0).to_numpy()
report = {
    "model": "ExtraTrees log1p(SI), 60 trees per model, 2 protocols x 3 seeds + supervised exact neighbor",
    "oof_summary": "canonical no-max=73927.60, no-top7=25791.78; scaffold no-max=227290.91, no-top7=20001.97",
    "exact_neighbor_rows": used,
    "manual_post_hoc_replacements": False,
    "local_reconstructed_diagnostic": metrics(si_true, pred),
    "independent_evaluation": False,
    "official_kaggle_score_unknown": True,
    "test_concentrations_used_for_prediction": False,
    "submission": "submission_clean_et_log.csv",
}
(OUT / "et_log_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))


In [10]:
# Ensemble of the public hybrid and clean ExtraTrees
import json

def metric_values(target, pred):
    err = (target - pred) ** 2
    keep = np.arange(len(target)) != np.argmax(target)
    keep7 = np.ones(len(target), bool)
    keep7[np.argsort(target)[-7:]] = False
    return {"mse": float(err.mean()), "mse_no_max": float(err[keep].mean()),
            "mse_no_top7": float(err[keep7].mean()), "mae": float(np.abs(target - pred).mean())}

et_oof = np.mean(list(oof_predictions.values()), axis=0)
rows = []
for v2_weight in [0, .25, .5, .75, 1.0]:
    candidate = v2_weight * pred_public + (1 - v2_weight) * pred
    # Selection uses only original train OOF, not test predictions.
    candidate_oof = v2_weight * oof + (1 - v2_weight) * et_oof
    rows.append({"v2_weight": v2_weight, **metric_values(y, candidate_oof)})
selection = pd.DataFrame(rows)
selection["full_rank"] = selection.mse.rank(pct=True)
selection["no_max_rank"] = selection.mse_no_max.rank(pct=True)
selection["no7_rank"] = selection.mse_no_top7.rank(pct=True)
selection["mean_rank"] = selection[["full_rank", "no_max_rank", "no7_rank"]].mean(axis=1)
selection = selection.sort_values("mean_rank")
best = selection.iloc[0]
v2_weight = float(best.v2_weight)
pred_ensemble = v2_weight * pred_public + (1 - v2_weight) * pred
pred_5050 = .5 * pred_public + .5 * pred

# Additional fixed variants: no local-test selection is performed.
all_fps = tr_fps + ext_fps
sim = np.array([max(DataStructs.BulkTanimotoSimilarity(query, all_fps)) for query in te_fps])
prior = float(np.median(y))
pred_conf1 = sim * pred + (1 - sim) * prior
pred_v2_2575 = .25 * pred_public + .75 * pred
pred_v2_7525 = .75 * pred_public + .25 * pred
pred_triple_equal = (pred_public + pred + pred_conf1) / 3
pred_triple_25_50_25 = .25 * pred_public + .5 * pred + .25 * pred_conf1
pred_v2_conf_5050 = .5 * pred_public + .5 * pred_conf1
pred_v2_conf_2575 = .25 * pred_public + .75 * pred_conf1
pred_v2_conf_7525 = .75 * pred_public + .25 * pred_conf1
pred_v2_conf_6040 = .6 * pred_public + .4 * pred_conf1
pred_v2_conf_4060 = .4 * pred_public + .6 * pred_conf1
pred_v2_conf_7030 = .7 * pred_public + .3 * pred_conf1
pred_v2_conf_3070 = .3 * pred_public + .7 * pred_conf1
pred_conf15 = sim ** 1.5 * pred + (1 - sim ** 1.5) * prior
pred_conf2 = sim ** 2 * pred + (1 - sim ** 2) * prior
pred_v2_conf15_5050 = .5 * pred_public + .5 * pred_conf15
pred_v2_conf2_5050 = .5 * pred_public + .5 * pred_conf2
fixed_variants = {
    "v2_et_2575": pred_v2_2575,
    "v2_et_7525": pred_v2_7525,
    "triple_equal": pred_triple_equal,
    "triple_25_50_25": pred_triple_25_50_25,
    "v2_conf_5050": pred_v2_conf_5050,
    "v2_conf_2575": pred_v2_conf_2575,
    "v2_conf_7525": pred_v2_conf_7525,
    "v2_conf_6040": pred_v2_conf_6040,
    "v2_conf_4060": pred_v2_conf_4060,
    "v2_conf_7030": pred_v2_conf_7030,
    "v2_conf_3070": pred_v2_conf_3070,
    "v2_conf15_5050": pred_v2_conf15_5050,
    "v2_conf2_5050": pred_v2_conf2_5050,
}
for name, values in fixed_variants.items():
    pd.DataFrame({"index": te["index"], "SI VACV": values}).to_csv(
        OUT / f"submission_{name}.csv", index=False)

pd.DataFrame({"index": te["index"], "SI VACV": pred_ensemble}).to_csv(
    OUT / "submission_ensemble_ncgc.csv", index=False)
pd.DataFrame({"index": te["index"], "SI VACV": pred_5050}).to_csv(
    OUT / "submission_ensemble_ncgc_5050.csv", index=False)
selection.to_csv(OUT / "ensemble_ncgc_oof_selection.csv", index=False)

si_true = (te.CC50_uM / te.IC50_uM_VACV).replace([np.inf, -np.inf], np.nan)
si_true = si_true.where(te.IC50_uM_VACV.notna(), 0.0).fillna(3.0).to_numpy()
report = {
    "v2_weight": v2_weight,
    "oof": metric_values(y, v2_weight * oof + (1 - v2_weight) * et_oof),
    "ensemble_local_diagnostic": metric_values(si_true, pred_ensemble),
    "ensemble_5050_local_diagnostic": metric_values(si_true, pred_5050),
    "fixed_variants": {name: metric_values(si_true, values) for name, values in fixed_variants.items()},
    "test_targets_used_for_fit_or_selection": False,
    "independent_evaluation": False,
    "selection": selection.to_dict(orient="records"),
}
(OUT / "ensemble_ncgc_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))